
# OLS — 편미분으로 유도하는 최소제곱 회귀

> 본 노트북의 목적은 **최소제곱법(Ordinary Least Squares, OLS)** 의 추정식을  
> "편미분 = 0" 이라는 한 줄에서 손으로 유도하고, 그 결과를 NumPy 로 직접 계산하여  
> scikit-learn 의 `LinearRegression` 과 비교하는 것이다.

기계학습 강의노트 · 04 · 부록 A.  
대상: 통계학 · 데이터과학 학부생.



## 1.  무엇을 최소화하는가

관측쌍 $(x_i, y_i)_{i=1}^{n}$ 에 대해 회귀 모형을

$$
y_i \;=\; \beta_0 \;+\; \beta_1 x_i \;+\; \delta_i, \qquad \mathrm{Var}(\delta_i) = \sigma^2
$$

로 두자.  OLS 는 **수직(Y축 방향) 잔차의 제곱합**

$$
S(\beta_0,\beta_1) \;=\; \sum_{i=1}^{n} \bigl(y_i - \beta_0 - \beta_1 x_i\bigr)^{2}
$$

을 최소화하는 $(\hat\beta_0, \hat\beta_1)$ 을 찾는다.

> **가정.** X 는 측정오차 없이 \"정확히\" 관측된 값이고, 오차는 오직 Y 에만 존재한다.  
> 이 가정이 깨지면 OLS 의 기울기는 0 쪽으로 편향된다(*attenuation bias*).



## 2.  편미분 = 0 을 푼다

$S$ 는 매끄러운 이차함수이므로 극값이 곧 최소값이다. 두 변수에 대해 편미분을 0 으로 놓는다.

$$
\frac{\partial S}{\partial \beta_0} \;=\; -2 \sum_i (y_i - \beta_0 - \beta_1 x_i) \;=\; 0
$$

$$
\frac{\partial S}{\partial \beta_1} \;=\; -2 \sum_i x_i (y_i - \beta_0 - \beta_1 x_i) \;=\; 0
$$

두 식을 정리하면 **정규방정식(normal equations)**

$$
\begin{aligned}
\sum_i y_i &= n\beta_0 + \beta_1 \sum_i x_i \\
\sum_i x_i y_i &= \beta_0 \sum_i x_i + \beta_1 \sum_i x_i^{2}
\end{aligned}
$$

가 얻어진다. 이를 풀면 닫힌 형태의 해

$$
\boxed{\;
\hat\beta_1 \;=\; \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2}, \qquad
\hat\beta_0 \;=\; \bar y - \hat\beta_1 \bar x
\;}
$$

가 된다. 다변량 일반화는 행렬 표기로

$$
\boldsymbol{y} = \boldsymbol{X}\boldsymbol{\beta} + \boldsymbol{\delta}, \qquad
\hat{\boldsymbol{\beta}} = (\boldsymbol{X}^{\mathsf T}\boldsymbol{X})^{-1}\boldsymbol{X}^{\mathsf T}\boldsymbol{y}.
$$



## 3.  코드로 확인한다

먼저 합성 자료를 만든다. 참값은 $\beta_0 = 3,\;\beta_1 = 2$ 이고 잡음의 표준편차는 1.5 이다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=20260515)

# 참값
beta0_true, beta1_true = 3.0, 2.0
n = 60
x = rng.uniform(0, 10, n)
y = beta0_true + beta1_true * x + rng.normal(0, 1.5, n)

print(f"x: shape={x.shape},  mean={x.mean():.3f}")
print(f"y: shape={y.shape},  mean={y.mean():.3f}")



### 3.1  닫힌 형태의 해를 직접 계산한다

위에서 유도한 식 $\hat\beta_1 = \sum(x_i - \bar x)(y_i - \bar y) / \sum(x_i - \bar x)^2$ 그대로이다.


In [ ]:
x_bar = x.mean()
y_bar = y.mean()

beta1_hat = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
beta0_hat = y_bar - beta1_hat * x_bar

print(f"수동 계산:  β̂₀ = {beta0_hat:.4f}   β̂₁ = {beta1_hat:.4f}")
print(f"참값:        β₀  = {beta0_true:.4f}   β₁  = {beta1_true:.4f}")



### 3.2  행렬식 $\hat{\boldsymbol{\beta}} = (\boldsymbol{X}^{\mathsf T}\boldsymbol{X})^{-1}\boldsymbol{X}^{\mathsf T}\boldsymbol{y}$ 로 풀기

설계행렬 $\boldsymbol{X}$ 의 첫 열을 1 로 두면 절편을 포함한 계수가 한 번에 나온다.  
실무에서는 수치 안정성을 위해 직접 역행렬을 만들지 않고 `np.linalg.lstsq` 를 쓴다.


In [ ]:
# 설계행렬 (n × 2)
X_design = np.column_stack([np.ones(n), x])

# 방법 1: 역행렬 직접
beta_hat_inv = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y
print(f"역행렬 풀이 :  β̂ = {beta_hat_inv}")

# 방법 2: 수치적으로 안전한 최소제곱 해
beta_hat_lstsq, *_ = np.linalg.lstsq(X_design, y, rcond=None)
print(f"lstsq 풀이  :  β̂ = {beta_hat_lstsq}")



### 3.3  scikit-learn 으로도 풀어 본다

세 방법이 모두 같은 답을 주어야 한다.


In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"sklearn   :  β̂₀ = {lr.intercept_:.4f}   β̂₁ = {lr.coef_[0]:.4f}")



### 3.4  잔차를 시각화한다

OLS 의 의미는 \"수직 잔차의 제곱합 최소화\" 이다. 각 점에서 회귀선까지 **수직 방향**으로 떨어진 거리가 잔차이다.


In [ ]:
xs = np.linspace(x.min() - 0.5, x.max() + 0.5, 100)
y_hat_line = beta0_hat + beta1_hat * xs

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(xs, y_hat_line, color="#1E2761", linewidth=2.5, label="OLS fit")
# 수직 잔차
for xi, yi in zip(x, y):
    yi_hat = beta0_hat + beta1_hat * xi
    ax.plot([xi, xi], [yi, yi_hat], color="#E0A11B", linewidth=1, alpha=0.7)
ax.scatter(x, y, color="#1B1F2A", s=30, edgecolor="white", linewidth=0.7, zorder=3)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("OLS — 수직 방향(Y축) 잔차의 제곱합을 최소화")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 4.  다변량으로 확장

설명변수가 둘 이상일 때도 같은 사고방식이 그대로 적용된다.  설계행렬 $\boldsymbol{X}$ 의 열에 변수를 쌓고 $\hat{\boldsymbol{\beta}} = (\boldsymbol{X}^{\mathsf T}\boldsymbol{X})^{-1}\boldsymbol{X}^{\mathsf T}\boldsymbol{y}$ 를 계산하면 된다.


In [ ]:
# 3개 설명변수
n = 200
X1 = rng.normal(0, 1, n)
X2 = rng.normal(0, 1, n)
X3 = rng.normal(0, 1, n)
y3 = 1.0 + 2.0 * X1 - 1.5 * X2 + 0.5 * X3 + rng.normal(0, 0.5, n)

# 설계행렬 (intercept + 3 변수)
X_design3 = np.column_stack([np.ones(n), X1, X2, X3])
beta_hat3, *_ = np.linalg.lstsq(X_design3, y3, rcond=None)

print(f"추정:  β̂ = {np.round(beta_hat3, 3)}")
print(f"참값:  β  = [1.0, 2.0, -1.5, 0.5]")



## 5.  OLS 의 가정과 한계

다음 가정이 성립할 때 OLS 의 해는 BLUE(Best Linear Unbiased Estimator)가 된다 — Gauss-Markov 정리.

1. **선형성**: 모형이 모수에 대해 선형.
2. **외생성**: $\mathbb{E}[\delta_i \mid X_i] = 0$.
3. **등분산성(homoskedasticity)**: $\mathrm{Var}(\delta_i) = \sigma^2$ 일정.
4. **무상관**: $\mathrm{Cov}(\delta_i, \delta_j) = 0$ for $i \neq j$.
5. **X 에 측정오차가 없다**.

가정 5 가 깨지면 OLS 는 **감쇠 편향** 을 일으킨다 — 다음 노트북(Deming) 에서 다룬다.

가정 3 이 깨지면 가중 최소제곱(WLS), 가정 4 가 깨지면 일반화 최소제곱(GLS) 로 일반화한다. 모두 \"편미분 = 0\" 이라는 같은 사고방식 위에 있다.



## 6.  연습문제

1. $n = 30$ 인 자료에서 $\hat\beta_1$ 의 분산이 $\sigma^2 / \sum(x_i - \bar x)^2$ 임을 유도하라.
2. 위 식이 알려주는 \"X 의 분포가 넓을수록 추정이 정확해진다\" 라는 직관을 한 문단으로 적어 보라.
3. 본 노트북의 코드에서 잡음 표준편차를 1.5 → 5 로 바꾸면 추정값이 얼마나 달라지는가? 직접 실험하라.
4. 다변량 코드에서 $X_1$ 과 $X_2$ 의 상관계수를 0.99 로 만들었을 때 추정값이 어떻게 변하는지 관찰하라. (이것이 다음 PCA 노트북의 \"다중공선성\" 문제이다.)
